Phase 0 : Mise en route

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Phase A : Prédire les prix immobiliers (régression)

In [ ]:

def charger_immobilier():
    """Charge California Housing, renvoie X, y.
    Doit afficher : nombre de lignes, nombre de variables, et l'unité de la cible.
    """
    # TODO : data = fetch_california_housing()
    data = fetch_california_housing()
    # TODO : afficher data.data.shape et data.feature_names
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    print(f"California Housing : {X.shape} variables, cible = prix médian en centaines de milliers de $")
    # TODO : renvoyer X, y
    return X, y

In [9]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.
    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    # TODO : fit, predict
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    # TODO : calculer r2_score, mean_absolute_error, root_mean_squared_error
    r2 = sklearn.metrics.r2_score(y_test, y_pred)
    mae = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred))
    # TODO : renvoyer {"r2": ..., "mae": ..., "rmse": ...}
    return {"r2": r2, "mae": mae, "rmse": rmse}

In [20]:
X, y = charger_immobilier()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = make_pipeline(StandardScaler(), LinearRegression())
rf = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))

res_lr = evaluer_regression(lr, X_train, X_test, y_train, y_test)
res_rf = evaluer_regression(rf, X_train, X_test, y_train, y_test)
print(f"LinearRegression : R2={res_lr['r2']:.2f} MAE={res_lr['mae']:.2f} RMSE={res_lr['rmse']:.2f}")
print(f"RandomForest     : R2={res_rf['r2']:.2f} MAE={res_rf['mae']:.2f} RMSE={res_rf['rmse']:.2f}")

California Housing : (20640, 8) variables, cible = prix médian en centaines de milliers de $
LinearRegression : R2=0.58 MAE=0.53 RMSE=0.75
RandomForest     : R2=0.81 MAE=0.33 RMSE=0.51


### Cas limite : entraînement sur 100 lignes

Oui, le **R² peut fortement baisser**. Avec seulement 100 lignes, le modèle a moins d'exemples pour apprendre et ses prédictions sont moins fiables.

**Pourquoi ?**
Un modèle a besoin de beaucoup de données pour comprendre les relations entre les variables et faire de bonnes prédictions.

---

### Cas adversarial : revenu médian = 0 et 9000 habitants

Le modèle peut renvoyer une valeur peu réaliste, car ces données sont très différentes de celles utilisées pour l'entraînement.

**En production :**
Il faut vérifier que les données d'entrée sont valides et refuser ou signaler les valeurs hors plage avant de faire la prédiction.


Phase B : Segmenter les clients d'AirBnB (non supervisé)

In [ ]:
def charger_airbnb(url_csv):
    df = pd.read_csv(url_csv)

    colonnes_utiles = ['price', 'minimum_nights', 'number_of_reviews', 'availability_365']
    colonnes_utiles = [col for col in colonnes_utiles if col in df.columns]
    X = df[colonnes_utiles].copy()

    for col in X.columns:
        if X[col].dtype == object:
            X[col] = pd.to_numeric(
                X[col].astype(str).str.replace(r'[^\d.]', '', regex=True),
                errors='coerce'
            )

    X = X.loc[:, X.isna().mean() < 0.8]
    print("Colonnes retenues après filtre NaN:", X.columns.tolist())

    X = X.dropna()
    print(f"Listings chargés : {X.shape[0]} lignes, {X.shape[1]} colonnes numériques retenues")
    return X

In [46]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    """Pour chaque k, renvoie inertie et silhouette.
    Doit afficher un tableau permettant de repérer le coude / le meilleur k.
    """
    resultats = []
    # TODO : pour chaque k, fit KMeans(n_init=10), récupérer inertia_ et silhouette_score
    for k in k_range:
        kmeans = sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=42)
        kmeans.fit(X_scaled)

        inertie = kmeans.inertia_
        silhouette = sklearn.metrics.silhouette_score(X_scaled, kmeans.labels_)
        resultats.append((k, inertie, silhouette))
        # TODO : afficher k, inertie, silhouette
        print(f"k={k} : inertie={inertie:.2f}, silhouette={silhouette:.2f}")
    
    k_retenu = max(resultats, key=lambda x: x[2])[0]
    print(f"nSegment retenu : {k_retenu}")        

In [47]:
url_csv = "data/listings.csv"
df_airbnb = charger_airbnb(url_csv)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_airbnb)

choisir_k(X_scaled)


Colonnes retenues après filtre NaN: ['minimum_nights', 'number_of_reviews', 'availability_365']
Listings chargés : 453 lignes, 3 colonnes numériques retenues
k=2 : inertie=988.69, silhouette=0.48
k=3 : inertie=699.52, silhouette=0.48
k=4 : inertie=429.99, silhouette=0.52
k=5 : inertie=319.44, silhouette=0.54
k=6 : inertie=248.85, silhouette=0.54
k=7 : inertie=210.76, silhouette=0.46
k=8 : inertie=179.69, silhouette=0.48
nSegment retenu : 5


### Cas normal

5 segments ont été identifiés. Chaque segment correspond à un groupe d'annonces ayant des caractéristiques similaires.

### Cas limite

Sans standardisation, les clusters ont moins de sens car une variable avec de grandes valeurs peut dominer les autres et influencer le résultat.

### Cas adversarial

Une annonce à 100 000 € déplace les centres des clusters et fausse le regroupement. C'est pourquoi le nettoyage des valeurs aberrantes est indispensable avant d'appliquer KMeans.


Phase C : Courriel vs spam (texte)

In [49]:
def vectoriser_textes(messages, vectorizer=None):
    """Transforme une liste de textes en matrice numérique.
    Si vectorizer est None, en crée un (fit) ; sinon réutilise (transform).
    Doit renvoyer (X, vectorizer).
    """
    # TODO : créer ou réutiliser le vectorizer
    # TODO : fit_transform ou transform selon le cas
    if vectorizer is None:
        vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(max_features=1000)
        X = vectorizer.fit_transform(messages)
    else:
        X = vectorizer.transform(messages)
    
    return X, vectorizer

In [50]:
def evaluer_spam(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie precision/recall/f1 de la classe spam.
    Doit afficher le classification_report (PAS juste l'accuracy).
    """
    # TODO : fit, predict
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    # TODO : afficher classification_report avec precision/recall/f1
    print(classification_report(y_test, y_pred))    
    pass

In [57]:
print("HAPPY PATH")

df_sms = pd.read_csv("spam_data/SMSSpamCollection", sep="\t", names=["label", "message"])

X_text = df_sms["message"]
y_label = df_sms["label"] 

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text, y_label, test_size=0.2, random_state=42, stratify=y_label
)

X_train_vec, mon_vectorizer = vectoriser_textes(X_train_t)
X_test_vec, _ = vectoriser_textes(X_test_t, vectorizer=mon_vectorizer)

nb_model = MultinomialNB()
evaluer_spam(nb_model, X_train_vec, X_test_vec, y_train_t, y_test_t)

print("EDGE CASE : Message vide")
# On teste un message vide ""
msg_vide_vec, _ = vectoriser_textes([""], vectorizer=mon_vectorizer)
print("Prédiction pour message vide :", nb_model.predict(msg_vide_vec)[0])

print("\nADVERSARIAL : Spam déguisé")
# On teste un spam déguisé en message amical
msg_piege = ["salut, ton colis t attend, confirme ici"]
msg_piege_vec, _ = vectoriser_textes(msg_piege, vectorizer=mon_vectorizer)
print("Prédiction pour le spam déguisé :", nb_model.predict(msg_piege_vec)[0])

HAPPY PATH
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       0.99      0.83      0.91       149

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

EDGE CASE : Message vide
Prédiction pour message vide : ham

ADVERSARIAL : Spam déguisé
Prédiction pour le spam déguisé : ham


### Happy path

Le modèle détecte correctement la majorité des spams. Le **recall spam est élevé**, donc il repère bien les messages indésirables.

---

### Edge case (message vide)

Un message vide est accepté par le vectorizer et est généralement classé comme **ham** (non spam).

---

### Adversarial (spam déguisé)

Le spam déguisé est mal classé en **ham**. Le modèle se fait donc piéger par des messages trop proches du langage normal.


Phase D : Décrypter les signaux d'un sonar (classification binaire)

In [58]:
def charger_sonar(url):
    """Charge le sonar, sépare X (60 colonnes) et y (M/R -> 1/0).
    Doit renvoyer X, y et afficher la répartition des classes.
    """
    # TODO : pd.read_csv(url, header=None)
    df = pd.read_csv(url, header=None)
    X = df.iloc[:, :-1]
    y_raw = df.iloc[:, -1]
    # TODO : dernière colonne = label (M=mine=1, R=rock=0)
    y = y_raw.map({'M': 1, 'R': 0})
    mines = (y == 1).sum()
    rochers = (y == 0).sum()

    # TODO : renvoyer X, y, afficher la répartition
    print(f"Sonar : ({df.shape[0]}, {df.shape[1]-1}), mines={mines}, rochers={rochers}")
    return X, y

In [67]:
url_sonar = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"
X, y = charger_sonar(url_sonar)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=42))
svc = make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=42))
rf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=42))

lr.fit(X_train, y_train)
svc.fit(X_train, y_train)
rf.fit(X_train, y_train)

print(f"LogisticRegression : accuracy={accuracy_score(y_test, lr.predict(X_test)):.2f}")
print(f"SVC (rbf)          : accuracy={accuracy_score(y_test, svc.predict(X_test)):.2f}")
print(f"RandomForest       : accuracy={accuracy_score(y_test, rf.predict(X_test)):.2f}")

Sonar : (208, 60), mines=111, rochers=97
LogisticRegression : accuracy=0.83
SVC (rbf)          : accuracy=0.93
RandomForest       : accuracy=0.83


### Cas normal

Le **SVM (rbf)** est le plus performant avec **0.93 d’accuracy**. Il est bien adapté à ce type de données avec beaucoup de variables et peu d’exemples.

---

### Cas limite (sans standardisation)

Sans standardisation, les performances du **SVM** et de la **régression logistique** baissent.
Le **Random Forest** est moins impacté car il dépend peu de l’échelle des variables.

---

### Cas adversarial (écho à zéro)

Un signal avec toutes les valeurs à zéro est quand même classé par le modèle avec une certaine confiance, même s’il n’a aucun sens réel.

En pratique, ce type de donnée devrait être détecté en amont comme une **erreur de capteur**, et ne jamais être envoyé au modèle.
